In [1]:
from loqs.backends import QSimQuantumState, PyGSTiNoiseModel
from loqs.core import QuantumProgram

from loqs.codepacks import d3_5q_code

import pygsti

In [2]:
code_5q = d3_5q_code.create_qec_code()

In [3]:
code_5q.instructions.keys()

dict_keys(['Non-FT Minus Prep', 'X', 'Z', 'H', 'Measure Physical Qubits'])

In [4]:
# Define a pyGSTi processor spec and noise model
# This is the only pyGSTi required here,
# and could eventually be traded out for something else
qubits = ["A0", "A1"] + [f"D{i+2}" for i in range(5)]
gate_names = ["Gxpi2", "Gypi2", "Gzpi2", "Gh", "Gcnot", "Gcphase"]

# TODO: Currently Iz does not need to be set here
# This is because QSimQuantumState actually does not try to pull the rep
# Otherwise, this would result in a KeyError in PyGSTiNoiseModel.get_reps(),
# since it technically should be provided
pspec = pygsti.processors.QubitProcessorSpec(
    len(qubits), 
    gate_names=gate_names,
    qubit_labels=qubits,
    availability={k: "all-combinations" for k in gate_names}
)

ideal_model_pygsti = pygsti.models.create_crosstalk_free_model(pspec)

ideal_model = PyGSTiNoiseModel(ideal_model_pygsti)


In [5]:
# Logical quantum program information
patch_types = {"5Q": code_5q}

# Stack
# The "Init *" instructions will be generated by the QuantumProgram based on the values
# of state_type and patch_types below
stack = [
    ("Init State", None, (len(qubits),), {"qubit_labels": qubits}), # Autogenerated from state_tyep
    ("Init Patch 5Q", None, ("L0", qubits)), # Autogenerated from patch_types. Note that 5Q here must be a key
    ("Non-FT Minus Prep", "L0"),
    ("X", "L0"),
    ("Measure Physical Qubits", "L0")
]

program = QuantumProgram(
    stack,
    default_noise_model=ideal_model,
    state_type=QSimQuantumState,
    patch_types=patch_types
)

In [6]:
print(stack)

[('Init State', None, (7,), {'qubit_labels': ['A0', 'A1', 'D2', 'D3', 'D4', 'D5', 'D6']}), ('Init Patch 5Q', None, ('L0', ['A0', 'A1', 'D2', 'D3', 'D4', 'D5', 'D6'])), ('Non-FT Minus Prep', 'L0'), ('X', 'L0'), ('Measure Physical Qubits', 'L0')]


In [7]:
print(program.history[-1]["stack"])

InstructionStack with 5 items:
  InstructionLabel(Init State,None,(7,),{'qubit_labels': ['A0', 'A1', 'D2', 'D3', 'D4', 'D5', 'D6']})
  InstructionLabel(Init Patch 5Q,None,('L0', ['A0', 'A1', 'D2', 'D3', 'D4', 'D5', 'D6']),{})
  InstructionLabel(Non-FT Minus Prep,L0,(),{})
  InstructionLabel(X,L0,(),{})
  InstructionLabel(Measure Physical Qubits,L0,(),{})



In [8]:
program.run()

Working on frame 1
Working on InstructionLabel(Init State,None,(7,),{'qubit_labels': ['A0', 'A1', 'D2', 'D3', 'D4', 'D5', 'D6']})
 with kwargs {'qubit_labels': ['A0', 'A1', 'D2', 'D3', 'D4', 'D5', 'D6']}

Resolved to <class 'loqs.core.instructions.instruction.Instruction'>: <loqs.core.instructions.instruction.Instruction object at 0x30fc182d0>
Applied frame: HistoryFrame with 2 Recordables:
  'state' (<class 'loqs.backends.state.qsimstate.QSimQuantumState'>):
    {'_state': <quantumsim.sparsedm.SparseDM object at 0x30fc633d0>}
  'instruction' (<class 'loqs.core.instructions.instruction.Instruction'>):

Final stored frame: HistoryFrame with 3 Recordables:
  'state' (<class 'loqs.backends.state.qsimstate.QSimQuantumState'>):
    {'_state': <quantumsim.sparsedm.SparseDM object at 0x30fc633d0>}
  'instruction' (<class 'loqs.core.instructions.instruction.Instruction'>):
  'stack' (<class 'loqs.core.instructions.instructionstack.InstructionStack'>):
    InstructionStack with 4 items:
      I

In [10]:
print(program.history)

History with 7 items:
  HistoryFrame with 0 Recordables:
  HistoryFrame with 1 Recordables:
    'stack' (<class 'loqs.core.instructions.instructionstack.InstructionStack'>):
      InstructionStack with 5 items:
        InstructionLabel(Init State,None,(7,),{'qubit_labels': ['A0', 'A1', 'D2', 'D3', 'D4', 'D5', 'D6']})
        InstructionLabel(Init Patch 5Q,None,('L0', ['A0', 'A1', 'D2', 'D3', 'D4', 'D5', 'D6']),{})
        InstructionLabel(Non-FT Minus Prep,L0,(),{'model': <loqs.backends.model.pygstimodel.PyGSTiNoiseModel object at 0x30f20a3d0>})
        InstructionLabel(X,L0,(),{'model': <loqs.backends.model.pygstimodel.PyGSTiNoiseModel object at 0x30f20a3d0>})
        InstructionLabel(Measure Physical Qubits,L0,(),{'model': <loqs.backends.model.pygstimodel.PyGSTiNoiseModel object at 0x30f20a3d0>})

  HistoryFrame with 3 Recordables:
    'state' (<class 'loqs.backends.state.qsimstate.QSimQuantumState'>):
      {'_state': <quantumsim.sparsedm.SparseDM object at 0x30fc633d0>}
    'instru

/Users/sserita/Documents/repos/loqs-public/loqs/core/frame.py:60: UserWarning: Accessing an expired object state. The returned object may actually belong to a future frame.
  warnings.warn(
